In [1]:
import pandas as pd
import re
import numpy as np
from collections import defaultdict
from sklearn.metrics import cohen_kappa_score
from analysis_util import fisher_metrics_with_optional_bootstrap, project_root

ROOT_DIR = project_root()

# print(f"Project root directory: {ROOT_DIR}")


In [2]:
df = pd.read_csv(
    ROOT_DIR / "data/model_grades/public_law/public_law_ground_truth_and_model_output.csv"
)

# df.head(2)

In [3]:
# --- parsing ---
gold_col = "MH_grade_mRPS"

_COL_RE = re.compile(
    r"^(?P<prefix>[^_]+)_prompt_(?P<prompt_id>.+?)__"
    r"(?P<model>.+?)__i(?P<iter>\d+)__"
    r"(?P<hash>h[0-9a-f]+|hc[0-9a-f]+)__"
    r"(?P<kind>.+)$"
)

def parse_col(col: str) -> dict:
    """
    Example:
    litellm_prompt_v1_ta_min_dp__gpt-5-2025-08-07__i1__h44136fa3__extracted_element
    """
    m = _COL_RE.match(col)
    if not m:
        raise ValueError(f"Unexpected column format: {col}")
    d = m.groupdict()
    d["iter"] = int(d["iter"])
    # prompt_family without engine prefix (litellm_/vllm_)
    d["prompt_family"] = f"prompt_{d['prompt_id']}"
    return d


# --- registry builder ---

def build_columns_registry(
    all_columns: list[str],
    *,
    skip_iters: set[int] = {4, 5},
    require_iters: tuple[int, ...] | None = (1, 2, 3),
) -> dict[tuple[str, str], list[str]]:
    """
    Returns:
      { (prompt_family, model): [col_i1, col_i2, col_i3, ...] }
    """
    tmp = defaultdict(dict)  # (prompt_family, model) -> {iter: col}

    for col in all_columns:
        d = parse_col(col)
        it = d["iter"]
        if it in skip_iters:
            continue

        key = (d["prompt_family"], d["model"])
        # if duplicates exist, last wins; adjust if you prefer to raise

        # Raise if duplicates exist (same key + iter)
        if it in tmp[key]:
            prev = tmp[key][it]
            raise ValueError(
                f"Duplicate column for key={key}, iter={it}: "
                f"existing={prev!r}, new={col!r}"
            )
        
        tmp[key][it] = col

    out = {}
    for key, by_iter in tmp.items():
        if require_iters is not None:
            missing = [i for i in require_iters if i not in by_iter]
            if missing:
                # If you prefer to keep partial sets, remove this block.
                continue

        out[key] = [by_iter[i] for i in sorted(by_iter.keys())]

    return out


# --- evaluation runner ---

def run_eval_suite(
    df,
    *,
    columns_registry: dict[tuple[str, str], list[str]],
    gold_col: str,
):
    results = {}  # (prompt_family, model) -> result dict
    for (prompt_family, model), cols in columns_registry.items():
        results[(prompt_family, model)] = fisher_metrics_with_optional_bootstrap(
            df,
            rater_cols=cols,
            reference_col=gold_col,
        )
    return results



def results_to_tidy_df(results: dict) -> pd.DataFrame:
    rows = []
    for key, r in results.items():
        # key is (prompt_family, model) OR (engine, prompt_family, model)
        if len(key) == 2:
            prompt_family, model = key
            engine = None
        else:
            engine, prompt_family, model = key

        label = f"{model}_{prompt_family}"

        if engine is not None:
            # optional: keep engine visible in label
            label = f"{engine}:{label}"

        rows.extend([
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "qwk",
                "mean_fisher": r["point_mean_qwk_fisher"],
                "sd_raw": r["point_sd_qwk_across_runs_kappa_space"],
            },
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "pearson",
                "mean_fisher": r["point_mean_pearson_fisher"],
                "sd_raw": r["point_sd_pearson_across_runs_raw_space"],
            },
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "spearman",
                "mean_fisher": r["point_mean_spearman_fisher"],
                "sd_raw": r["point_sd_spearman_across_runs_raw_space"],
            },
        ])

    return pd.DataFrame(rows)


# --- usage ---

# all_prompt_columns = [...]  # your long list from earlier
COLUMNS = build_columns_registry(df.filter(like="__extracted_element").columns.tolist())

results = run_eval_suite(df, columns_registry=COLUMNS, gold_col=gold_col)
tidy_mrps_df = results_to_tidy_df(results)



In [8]:
which_metric = "qwk"
df_f_f = (
    tidy_mrps_df.query(f'metric == "{which_metric}"')
           .sort_values(["prompt_family", "mean_fisher"], ascending=[True, False])
           .loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]]
           .round(3)
)


display(df_f_f)


,prompt_family,base_model,mean_fisher,sd_raw
3,prompt_v1_ta_min_dp,gpt-5.1-2025-11-13,0.596,0.043
0,prompt_v1_ta_min_dp,gpt-5-2025-08-07,0.576,0.030
33,prompt_v1_ta_min_dp,gemini-2.5-pro,0.568,0.149
42,prompt_v1_ta_min_dp,deepseek-v3.1-terminus,0.501,0.181
6,prompt_v1_ta_min_dp,gpt-5.2-2025-12-11,0.356,0.056
21,prompt_v1_ta_min_dp,gpt-4.1-2025-04-14,0.338,0.038
9,prompt_v1_ta_min_dp,gpt-4o-2024-11-20,0.283,0.036
228,prompt_v1_ta_min_dp,gpt-5-mini-2025-08-07,0.224,0.098
243,prompt_v1_ta_min_dp,qwen3-235b-a22b-thinking-2507,0.153,0.042
216,prompt_v1_ta_min_dp,mistral-large-2512,0.113,0.017


In [6]:
###
### Random baseline
###


# Reproducible RNG
rng = np.random.default_rng(seed=42)

n_baselines = 3
low, high = 0, 19  # high is exclusive

random_baselines = rng.integers(
    low=low,
    high=high,
    size=(len(df), n_baselines)
)

# Assign to DataFrame
for i in range(n_baselines):
    df[f"random_baseline_{i+1}"] = random_baselines[:, i]



random_baselines_col = ['random_baseline_1',
                        'random_baseline_2',
                        'random_baseline_3']

random_baseline=fisher_metrics_with_optional_bootstrap(df, rater_cols=random_baselines_col, reference_col=gold_col)
random_baseline

{'point_mean_qwk_fisher': 0.06812075089046421,
 'point_sd_qwk_across_runs_kappa_space': 0.3650001218729787,
 'point_mean_pearson_fisher': 0.144253009183738,
 'point_sd_pearson_across_runs_raw_space': 0.5108713441778256,
 'point_mean_spearman_fisher': 0.06855827453653814,
 'point_sd_spearman_across_runs_raw_space': 0.5237074134253424}

In [7]:
###
### calculating QWK for " KI-Unterstützung und Rohpunkteschemata: Die Zukunft der juristischen Klausurkorrektur? "
###

LABELS = list(range(19))
def _qwk(a: pd.Series, b: pd.Series) -> float:
    return float(cohen_kappa_score(a, b, labels=LABELS, weights="quadratic"))


###
### DW Results QWK Calculation
###

mh_DW_gemini_2_5_pro_mRPS = _qwk(
    df["MH_grade_mRPS"],
    df["DW_gemini_2_5_pro_mRPS"]
)

mh_DW_gemini_2_5_pro_oRPS = _qwk(
    df["MH_grade_mRPS"],
    df["DW_gemini_2_5_pro_oRPS"]
)   

mh_DW_gpt_4o_mRPS = _qwk(
    df["MH_grade_mRPS"],
    df["DW_gpt_4o_mRPS"]
    )

mh_DW_gpt_4o_oRPS = _qwk(
    df["MH_grade_mRPS"],
    df["DW_gpt_4o_oRPS"]
    )



###
### KK Results QWK Calculation
###

mh_KK_gemini_2_5_pro_mRPS = _qwk(
    df["MH_grade_mRPS"],
    df["KK_gemini_2_5_pro_mRPS"]
)

mh_KK_gemini_2_5_pro_oRPS = _qwk(
    df["MH_grade_mRPS"],
    df["KK_gemini_2_5_pro_oRPS"]
)

mh_KK_gpt_4o_mRPS = _qwk(
    df["MH_grade_mRPS"],
    df["KK_gpt_4o_mRPS"]
)

mh_KK_gpt_4o_oRPS = _qwk(
    df["MH_grade_mRPS"],
    df["KK_gpt_4o_oRPS"]
)

import pandas as pd

results = pd.DataFrame(
    [
        # DW
        {"Group": "DW", "Model": "gemini_2_5_pro", "mRPS": mh_DW_gemini_2_5_pro_mRPS, "oRPS": mh_DW_gemini_2_5_pro_oRPS},
        {"Group": "DW", "Model": "gpt_4o",         "mRPS": mh_DW_gpt_4o_mRPS,         "oRPS": mh_DW_gpt_4o_oRPS},

        # KK
        {"Group": "KK", "Model": "gemini_2_5_pro", "mRPS": mh_KK_gemini_2_5_pro_mRPS, "oRPS": mh_KK_gemini_2_5_pro_oRPS},
        {"Group": "KK", "Model": "gpt_4o",         "mRPS": mh_KK_gpt_4o_mRPS,         "oRPS": mh_KK_gpt_4o_oRPS},
    ]
)


results = (
    results
    .set_index(["Group", "Model"])
    .sort_index()
    .round(3)
)

display(results)



mRPS   oRPS
Group Model                       
DW    gemini_2_5_pro  0.672  0.125
      gpt_4o          0.716  0.395
KK    gemini_2_5_pro  0.764 -0.113
      gpt_4o          0.464  0.327